<h1 style="color:#d53a49;font-weight:600;">Time series forecasting with ARIMA</h1>

---

In this notebook, we demonstrate how to:
- prepare time series data for training an **ARIMA** time series forecasting model
- implement a simple ARIMA model to forecast the next HORIZON steps ahead (time *t+1* through *t+HORIZON*) in the time series
- evaluate the model 

The data in this example is taken from the GEFCom2014 forecasting competition. It consists of 3 years of hourly electricity load and temperature values between 2012 and 2014. The task is to forecast future values of electricity load. In this example, we show how to forecast one time step ahead, using historical load data only.

<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">General Concepts</h3>

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">Stationarity</span><br/>
Data whose statistical properties, such as mean and variance, do not change over time. Non-stationary data shows fluctuations due to trends and/or seasonality.

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">Differencing</span><br/>
Transforming non-stationary data to make it stationary by eliminating trend and seasonality, thus stabilizing the mean of the time series.

<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">ARIMA in the context of time series</h3>

- **AR (AutoRegressive)** : The evolving variable of interest is regressed on its prior values (lags)
- **I (Integrated)** : The data values have been replaced by the difference between each value and the previous value.
- **MA (Moving Average)** : The regression error is a linear combination of terms whose values occured at various times in the past.

<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">Build an ARIMA Model</h3>

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
import math
import statsmodels

from pandas.plotting import autocorrelation_plot
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from common.utils import load_data, mape
from IPython.display import Image

pd.options.display.float_format = "{:,.2f}".format
np.set_printoptions(precision=2)
warnings.filterwarnings("ignore")

Load the data intoa Pandas dataframe and take a look:

In [ ]:
energy = load_data("./data")[["load"]]
energy.head(10)

Plot all the available energy data, from January 2012 to December 2014:

In [ ]:
energy.plot(y="load", subplots=True, figsize=(15, 8), fontsize=12)
plt.xlabel("timestamp", fontsize=12)
plt.ylabel("load", fontsize=12)

plt.show()

<h4 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">Create training and testing datasets</h4>

The steps to follow are:
- Separate the data into training and test data.
- Train the model on the training data.
- Evaluate the model's accuracy using the test set.<br/>
    **NOTE**: The test set must cover a later period in time than the train set, so that the model doesn't gain information from future data!

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">1. SEPARATE DATA - VISUALIZE THE DIFFERENCES</span>

For the train data use: 01 Nov 2014 up to 29 Dec 2014<br/>
For the test data use : 30 Dec 2014 up to the end


In [ ]:
# fmt: off
train_start_dt="2014-11-01 00:00:00"
test_start_dt="2014-12-30 00:00:00"

(
    energy[(energy.index < test_start_dt) & (energy.index >= train_start_dt)][["load"]]
        .rename(columns={"load":"train"})
        .join(energy[test_start_dt:][["load"]]
              .rename(columns={"load":"test"}), how="outer",)
        .plot(y=["train","test"], figsize=(15,8), fontsize=12,)
)

plt.show()

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">2. Prepare the data for training</span>

Perform:

- **Filtering** : only include the time periods and columns we need.
- **Scaling** : Ensure the data is projected in the interval [0, 1].

In [ ]:
train = energy.copy()[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][["load"]]
test = energy.copy()[energy.index >= test_start_dt][["load"]]

print(f"Training data shape : {train.shape}")
print(f"Testing data shape : {test.shape}")

In [ ]:
scaler = MinMaxScaler()
train["load"] = scaler.fit_transform(train)
train.head(10)

Visualize the original train vs. scaled train data and notice that the distribution is exactly the same!

In [ ]:
# fmt: off
(
    energy[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][["load"]]
        .rename(columns={"load":"original load"})
        .plot.hist(bins=100,fontsize=12)
)
(
    train.rename(columns={"load": "scaled load"})
        .plot.hist(bins=100,fontsize=12)
)

plt.show()

Also, scale the test data:

In [ ]:
test["load"] = scaler.transform(test)
test.head()

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">3. Implement ARIMA</span>

The steps to follow are:
- Define the model by calling `SARIMAX()` and passing in the model parameters p, d and q.
- Train the model by calling `fit()` method.
- Make predictions with the `forecast()` method specifying the number of steps (`horizon`).  

**PARAMETERS OF THE ARIMA MODEL**

**p** : is associated with the autoregressive part of the model (AR) - past values.<br/>
**d** : is associated with the integrated part of the model (I) - controls the differencing.<br/>
**q** : is associated with the moving average part of the model (MA).

When using the seasonal ARIMA model, the parameters to use are: P, D and Q.

Specify the number of steps to forecast ahead e.g. 3 hours

In [ ]:
HORIZON = 3
print(f"Forecasting horizon : {HORIZON} hours")

Next, try some manual selections to find a good model:

| Parameter | Meaning |
|:----------|:--------|
| **p=4** | The current value depends on the last 4 hours |
| **d=1** | Differencing takes the first difference to remove the trend $y_{t} - y_{t-1}$ |
| **q=0** | No moving average terms |
| **P=1** | Seasonal AR: current value depends on the last value 24 hours ago |
| **D=1** | Seasonal Differencing: Takes $y_t - y_{t-24}$ |
| **Q=0** | Seasonal MA: None
| **s=24** | Period: hourly data with a daily cycle, so 24 steps per season |

In [ ]:
order = (4, 1, 0)
seasonal_order = (1, 1, 0, 24)

model = SARIMAX(endog=train, order=order, seasonal_order=seasonal_order)
results = model.fit()

print(results.summary())

<span style="font-size:1.2rem;font-weight:600;color:#d53a49;text-transform:uppercase;">4. Evaluate the model</span>

We'll use the `walk forward validation`:
- Train the model on the train dataset.
- Make a prediction on the next time step.
- Evaluate the prediction against the known value.
- Expand the train set to include the known value and repeat.
- Every time we add a new observation to the train set, remove the observation from the beginning of the set, to keep the training set window fixed.

**CREATE A TEST DATA POINT FOR EACH HORIZON STEP**

In [ ]:
test_shifted = test.copy()

for t in range(1, HORIZON + 1):
    test_shifted["load+" + str(t)] = test_shifted["load"].shift(-t, freq="h")

test_shifted = test_shifted.dropna(how="any")
test_shifted.head(5)

**MAKE PREDICTIONS ON YOUR TEST DATA**

Use the sliding window approach in a loop the size of the test data length:

In [ ]:
%%time
training_window=720

train_ts=train["load"]
test_ts=test_shifted

history=[x for x in train_ts]
history=history[(-training_window):]

predictions=list()

order=(2,1,0)
seasonal_order=(1,1,0,24)

for t in range(test_ts.shape[0]):
    obs=list(test_ts.iloc[t])
    history.append(obs[0])
    history.pop(0)

    model=SARIMAX(endog=history,order=order,seasonal_order=seasonal_order)
    model_fit=model.fit()
    yhat=model_fit.forecast(steps=HORIZON)
    predictions.append(yhat)

    print(test_ts.index[t])
    print(t+1,": predicted =",yhat,"expected =",obs)

**COMPARE THE PREDICTIONS TO THE ACTUAL LOAD**

In [ ]:
eval_df = pd.DataFrame(predictions, columns=["t+" + str(t) for t in range(1, HORIZON + 1)])
eval_df["timestamp"] = test.index[0 : len(test.index) - HORIZON]
eval_df = pd.melt(eval_df, id_vars="timestamp", value_name="prediction", var_name="h")
# Use only load+1, load+2, and load+3; drop the "load" column (time t)
actual_cols = ["load+" + str(t) for t in range(1, HORIZON + 1)]
eval_df["actual"] = np.transpose(test_ts[actual_cols]).to_numpy().ravel()
eval_df[["prediction", "actual"]] = scaler.inverse_transform(eval_df[["prediction", "actual"]])

eval_df.head()

**CHECK MODEL ACCURACY**

Test the mean absolute percentage error (MAPE) over all the predictions.

In [ ]:
if HORIZON > 1:
    eval_df["APE"] = (eval_df["prediction"] - eval_df["actual"]).abs() / eval_df["actual"]
    print(eval_df.groupby("h")["APE"].mean())

Calculate one step's MAPE:

In [ ]:
one_step_mape = mape(eval_df[eval_df["h"] == "t+1"]["prediction"], eval_df[eval_df["h"] == "t+1"]["actual"]) * 100

print(f"One step forecast MAP: {one_step_mape}")

Multi-step forecast MAPE:

In [ ]:
print(f"Multi-step forecast MAPE: {mape(eval_df.prediction,eval_df.actual)*100}")

**PLOT THE ACCURACY MEASUREMENTS**

In [ ]:
if HORIZON == 1:
    # Plotting single step forecast
    eval_df.plot(x="timestamp", y=["actual", "prediction"], style=["r", "b"], figsize=(15, 8))
else:
    # Plotting multi-step forecast
    plot_df = eval_df[(eval_df.h == "t+1")][["timestamp", "actual"]]

    for t in range(1, HORIZON + 1):
        plot_df["t+" + str(t)] = eval_df[(eval_df.h == "t+" + str(t))]["prediction"].values

    fig, ax = plt.subplots(1, 1, figsize=(15, 8))
    ax.plot(plot_df.timestamp, plot_df.actual, color="red", linewidth=4.0)

    for t in range(1, HORIZON + 1):
        x = plot_df.timestamp[(t - 1) :]
        y = plot_df["t+" + str(t)][0 : len(x)]
        ax.plot(x, y, color="blue", linewidth=4 * math.pow(0.9, t), alpha=math.pow(0.8, t))

    ax.legend(loc="best")

plt.xlabel("timestamp", fontsize=12)
plt.ylabel("load", fontsize=12)

plt.show()

<span style="font-size:1.4rem;font-weight:600;font-style:italic;color:#d53a49;text-decoration:underline;">References:</span>

- [Autoregressive Integrated Moving Average (ARIMA) - Wikipedia](https://en.wikipedia.org/wiki/Autoregressive_integrated_moving_average)
- [Stationary Process](https://en.wikipedia.org/wiki/Stationary_process)
- [Evaluating Forecast Accuracy](https://otexts.com/fpp2/accuracy.html)